## Vista general de todas las fuentes

In [0]:
%sql
-- Paso 1 (estructura), mirada global: cantidad de columnas de cada tabla bronze
-- y sus primeras 6, para ubicar dónde está la columna de país, de año, etc.
SELECT table_name,
       COUNT(*) AS columnas,
       CONCAT_WS(', ', SLICE(ARRAY_SORT(COLLECT_LIST(STRUCT(ordinal_position, column_name))).column_name, 1, 6)) AS primeras_6
FROM happiness.information_schema.columns
WHERE table_schema = 'bronze'
GROUP BY table_name
ORDER BY table_name;

## EDA Banco Mundial
### Paso 1

In [0]:
%sql
-- Banco Mundial · Paso 1: estructura. Esperamos 398.468 filas (D14).
SELECT COUNT(*) AS filas FROM happiness.bronze.banco_mundial

-- Banco Mundial · Paso 1: tipos. Todo tiene que ser STRING (D10/D15).
DESCRIBE TABLE happiness.bronze.banco_mundial

-- Banco Mundial · Paso 1: cómo se ven los datos.
SELECT * FROM happiness.bronze.banco_mundial LIMIT 5

**Paso 1 de Banco Mundial cerrado**: 398.468 filas (coincide con D14), 73 columnas (4 de clave, 66 de años de 1960 a 2025, y _rescued_data, _ingest_ts y _source), y todo es STRING salvo _ingest_ts (D10/D15 ✓). 
Dos cosas a considerar: los agregados regionales (AFE, por ejemplo) vienen mezclados con los países, y existe _rescued_data, que tiene que estar vacía. Esto último lo chequeamos ahora.

### Paso 2: 
- Nulos
- Nulos por año (temporal)
- Rango por indicador
- Recuento de valores que no se pueden convertir a número
- Cardinalidad y duplicados por clave de unicidad (país + indicador)

In [0]:
%sql
-- Distingue NULL real de texto 'null' en un año cualquiera (2020),
-- y verifica que las columnas clave no tengan nulos y que _rescued_data esté vacía
SELECT COUNT(*)                                AS filas,
       COUNT_IF(`2020` IS NULL)                AS nulo_real_2020,
       COUNT_IF(`2020` = 'null')               AS texto_null_2020,
       COUNT_IF(`2020` = '')                   AS vacio_2020,
       COUNT_IF(`Country Code` IS NULL)        AS sin_country_code,
       COUNT_IF(`Indicator Code` IS NULL)      AS sin_indicator_code,
       COUNT_IF(_rescued_data IS NOT NULL)     AS filas_rescatadas
FROM happiness.bronze.banco_mundial;

In [0]:
%sql
-- TEMP VIEW: vive solo en esta sesión del notebook, no crea tabla.
-- UNPIVOT con INCLUDE NULLS conserva las celdas vacías, que es justo lo que queremos contar.
CREATE OR REPLACE TEMP VIEW bm_largo AS
SELECT `Country Code` AS codigo_pais, `Indicator Code` AS codigo_ind, anio, valor
FROM happiness.bronze.banco_mundial
UNPIVOT INCLUDE NULLS (valor FOR anio IN (
  `1960`, `1961`, `1962`, `1963`, `1964`, `1965`, `1966`, `1967`, `1968`, `1969`, `1970`, `1971`, `1972`, `1973`, `1974`, `1975`, `1976`, `1977`, `1978`, `1979`, `1980`, `1981`, `1982`, `1983`, `1984`, `1985`, `1986`, `1987`, `1988`, `1989`, `1990`, `1991`, `1992`, `1993`, `1994`, `1995`, `1996`, `1997`, `1998`, `1999`, `2000`, `2001`, `2002`, `2003`, `2004`, `2005`, `2006`, `2007`, `2008`, `2009`, `2010`, `2011`, `2012`, `2013`, `2014`, `2015`, `2016`, `2017`, `2018`, `2019`, `2020`, `2021`, `2022`, `2023`, `2024`, `2025`
));

In [0]:
%sql
-- % de celdas con dato por año para los indicadores incluidos en config_indicadores.
-- "Con dato" = ni NULL real ni texto 'null'. Sirve para ver desde qué año la serie es usable.
SELECT l.anio,
       COUNT(*)                                                         AS celdas,
       COUNT_IF(l.valor IS NOT NULL AND l.valor <> 'null')              AS con_dato,
       ROUND(100 * COUNT_IF(l.valor IS NOT NULL AND l.valor <> 'null') / COUNT(*), 1) AS pct_con_dato
FROM bm_largo l
JOIN happiness.meta.config_indicadores c
  ON c.fuente = 'BANCO_MUNDIAL' AND c.incluido AND c.codigo = l.codigo_ind
GROUP BY l.anio
ORDER BY l.anio;

In [0]:
%sql
SELECT l.codigo_ind, c.nombre, c.unidad,
       MIN(TRY_CAST(l.valor AS DOUBLE)) AS minimo,
       MAX(TRY_CAST(l.valor AS DOUBLE)) AS maximo,
       COUNT_IF(l.valor IS NOT NULL AND l.valor <> 'null' AND TRY_CAST(l.valor AS DOUBLE) IS NULL) AS no_numericos,
       MIN(CASE WHEN l.valor <> 'null' THEN l.anio END) AS primer_anio,
       MAX(CASE WHEN l.valor <> 'null' THEN l.anio END) AS ultimo_anio
FROM bm_largo l
JOIN happiness.meta.config_indicadores c
  ON c.fuente = 'BANCO_MUNDIAL' AND c.incluido AND c.codigo = l.codigo_ind
GROUP BY ALL
ORDER BY l.codigo_ind;

In [0]:
%sql
-- Cardinalidad: si hay más nombres que códigos (o al revés), un mismo país o indicador
-- aparece escrito de dos formas. Duplicados: rn > 1 indica una clave repetida.
WITH numerado AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY `Country Code`, `Indicator Code` ORDER BY _ingest_ts) AS rn
  FROM happiness.bronze.banco_mundial
)
SELECT COUNT(DISTINCT `Country Code`)   AS codigos_pais,
       COUNT(DISTINCT `Country Name`)   AS nombres_pais,
       COUNT(DISTINCT `Indicator Code`) AS codigos_ind,
       COUNT(DISTINCT `Indicator Name`) AS nombres_ind,
       COUNT_IF(rn > 1)                 AS claves_duplicadas
FROM numerado;

## Hallazgos:

**Nulos:** en 2020 hay 181.215 NULL reales (45%) y 0 con el texto "null". Acá el CSV ya trae las celdas vacías, así que en Silver alcanza con el CAST.

**Cobertura por año:** 81% a 88% entre 2000 y 2021, y baja a 81% en 2022, 78% en 2023, 67% en 2024 y 40% en 2025. Los últimos años dependen del arrastre de 2 años (D3).

**Indicadores que no llegan al final del panel:** EG.FEC.RNEW.ZS (energía renovable) termina en 2022 y SH.STA.SUIC.P5 (suicidio) en 2021. Con arrastre de 2 años cubren hasta 2024 y 2023. Si el panel WHR llega a 2025 quedan sin dato al final. A revisar cuando veamos el rango de años de WHR.

**Valores extremos a revisar:** inflación hasta 23.773% (mínimo -17,6%), comercio 863% del PIB, gasto militar 117% del PIB, CO2 per cápita 202 t, celulares 574 por 100 personas, PIB -64% y esperanza de vida mínima de 11 años. Algunos son plausibles (economías muy chicas, guerras, hiperinflación). Para decidir hay que ver qué país-año los produce.

**Agregados mezclados:** los 266 códigos incluyen regiones y grupos de ingreso (como AFE). Los separamos al armar dim_pais_base.

In [0]:
%sql
select * from bm_largo

In [0]:
%sql
-- Paso 4 (outliers): top 3 país-año por indicador en los que mostraron rangos raros.
WITH numerico AS (
  SELECT l.codigo_ind, c.nombre, b.`Country Name` AS pais, l.codigo_pais, l.anio,
         TRY_CAST(l.valor AS DOUBLE) AS v
  FROM bm_largo l
  JOIN happiness.bronze.banco_mundial b
    ON b.`Country Code` = l.codigo_pais AND b.`Indicator Code` = l.codigo_ind
    JOIN happiness.meta.config_indicadores c
  ON c.fuente = 'BANCO_MUNDIAL' AND c.incluido AND c.codigo = l.codigo_ind
  WHERE l.codigo_ind IN ('FP.CPI.TOTL.ZG','NE.TRD.GNFS.ZS','MS.MIL.XPND.GD.ZS',
                         'EN.GHG.CO2.PC.CE.AR5','IT.CEL.SETS.P2','NY.GDP.MKTP.KD.ZG')
    AND l.valor IS NOT NULL
),
ranking AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY codigo_ind ORDER BY v DESC) AS rn FROM numerico
)
SELECT codigo_ind, nombre, pais, codigo_pais, anio, v FROM ranking WHERE rn <= 3 ORDER BY codigo_ind, rn;

Los extremos de Banco Mundial son todos reales o explicables, así que no hay que limpiar nada en Silver por esto. Son "valores atípicos revisados, se conservan":

- Palau, CO2 per cápita (hasta 203 t): economía muy chica con poca población.
- Congo (RD) 1994, Bolivia 1985 y Perú 1990, inflación: hiperinflaciones históricas.
- Fiji 2024 (574), Macao y Hong Kong, celulares por 100 personas: muchas líneas por persona (turismo, doble SIM).
- Kuwait 1991, gasto militar (117% del PIB): guerra del Golfo, con PIB colapsado.
- Islas Vírgenes de EE.UU. 2008, comercio (863% del PIB): economía chica y muy abierta.
- Guinea Ecuatorial 1997, Liberia 1997 y Libia 2012, crecimiento del PIB (hasta 150%): arranque del petróleo o rebote tras la guerra.